[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/estimation/odometry/odometry.ipynb)

# The Most Likely Trajectory from Odometry

This example computes the most likely trajectory of a robot around a lap and the uncertainty of every pose from noisy odometry readings and a loop-closing reading. The robot drives a lap and reads, at every stop, how far it moved since the stop before. Each step reading is slightly off, so chaining them by dead reckoning drifts; when the robot returns to the start and recognises it, one more relative pose reading links the end of the lap back to the start and closes the loop.

A pose is a motor, an error in it a twist on its right, and what pulls on a pose a forque, which pairs with a twist, `forque & twist`, into the change the twist makes. A reading's uncertainty is a covariance, a map from forques to twists, `Twist <- Forque`, and its inverse is the information that weighs a mismatch; a motor carries both between frames by its sandwich. Every reading pulls only on the two poses it links, so the Gauss-Newton operator is applied reading by reading, with no system matrix assembled, and conjugate gradients solve it both for the pose corrections and for every pose's marginal covariance.

In the matrix notation of pose-graph nonlinear least squares on $\mathrm{SE}(2)$, each reading $z_k$ of the motion from pose $x_i$ to pose $x_j$ defines a Lie-algebra residual $r_k = \log(z_k^{-1} x_i^{-1} x_j)^\vee$, the objective is $\tfrac{1}{2} \sum_k r_k^{\mathsf T} \Sigma_k^{-1} r_k$, and each Gauss-Newton step solves the normal equations $(J^{\mathsf T} W J)\, \delta = -J^{\mathsf T} W r$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, concatenate
from numga.algebras import PGA2D as ga
from examples.util.animation import save_figure, show_animation
from examples.estimation.odometry import render

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
Motor = ga.gatype.rotor()                           # Rotor+
Twist = ga.gatype.bivector()                        # Bivector+
# What pulls on a pose: a forque, which paired with a twist, `forque & twist`, is the change the twist makes.
Forque = ga.gatype.antibivector()                   # Antibivector-
Point = ga.gatype.antivector()                      # Antivector-
Plane = ga.gatype.vector()                          # Vector+
Covariance = ga.gatype((Twist, Forque))
Information = ga.gatype((Forque, Twist))
Quadric = ga.gatype((Plane, Point))
# The point each pose carries: the origin, dual to the weight direction w.
origin = mv.w.dual()                                # [] Point-

## 1. The lap and its readings

The robot drives one lap in 36 steps of 0.7 m, turning faster and slower twice a lap and slipping sideways one way and back twice a lap so the true loop closes. Each reading is a relative pose, the pose at its head as read from the pose at its tail, corrupted by a twist perturbation on the right drawn from the reading's covariance map `Twist <- Forque`. The readings cover every step to 3 cm and 0.01 rad, followed by one loop-closing reading of the last pose from the first pose to a millimetre. Every pose also carries a prior covariance: the first pose is known closely, while the remaining poses have only vague priors.

In [ ]:
n_steps = 36
n_poses = n_steps + 1
step_length = 0.7
# How much the turn rate varies, and how far the robot slips sideways, twice a lap.
turn_variation, slip = 0.4, 0.1
# How far a step reading and the closing reading are trusted, how well the first pose is known, and
# how vaguely the others are, as translation and rotation standard deviations.
reading_std, closing_std = (0.03, 0.01), (0.001, 0.0003)
known_std, vague_std = (0.01, 0.01), (100.0, 30.0)
# How far out the uncertainty ellipsoids lie, in standard deviations.
sigmas = 2.0

In [ ]:
def isotropic(translation_std: float, rotation_std: float) -> Covariance:
    """A body-frame covariance is isotropic in translation and in rotation about the origin."""
    basis_twists = mv(Twist, np.eye(len(Twist.output_subspace)))                # [modes] Twist+
    # A basis twist generates a rotation if its reverse product is one, and a translation if it is zero.
    rotational = basis_twists.scalar_norm_squared()                             # [modes] Scalar+
    variances = translation_std**2 + (rotation_std**2 - translation_std**2) * rotational  # [modes] Scalar+
    return (basis_twists * (basis_twists & Forque) * variances).sum(axis=0)     # [] Twist+ <- Forque-


def modes(uncertainty: Covariance) -> Twist:
    """The modes of a covariance are twists whose outer products sum back to it."""
    spread = Forque & uncertainty                                               # [...] Scalar+ <- (Forque-, Forque-)
    _, forques = spread.eigh(spread)                                            # [..., modes] Forque-
    return uncertainty[..., None](forques)                                      # [..., modes] Twist+


rng = np.random.default_rng(9)
phase = 2 * np.pi * np.arange(n_steps) / n_steps
turn = mv.xy * (1 + turn_variation * np.sin(2 * phase)) * 2 * np.pi / n_steps   # [steps] Twist+
sideways = mv.yw * slip * np.cos(2 * phase)                                     # [steps] Twist+
steps = ((turn + sideways + mv.xw * step_length) * 0.5).exp()                   # [steps] Motor+
# Each step composes on the right in the robot's own frame; a running right product equals the
# reverse of the running left product of the reverses.
truth = concatenate([mv.rotor()[None], steps.reverse().cumprod(axis=0).reverse()])  # [poses] Motor+

# The pose each reading leaves from and arrives at: every step, then the last pose from the first.
tails = np.append(np.arange(n_steps), 0)                                        # [readings]
heads = np.append(np.arange(1, n_poses), n_steps)                               # [readings]
noises = concatenate([isotropic(*reading_std) * np.ones(n_steps), isotropic(*closing_std)[None]])  # [readings] Twist+ <- Forque-
noise_modes = modes(noises)                                                     # [readings, modes] Twist+
errors = (noise_modes * rng.normal(size=noise_modes.shape)).sum(axis=-1)        # [readings] Twist+
readings = (truth[tails].inverse() * truth[heads]) * (errors * 0.5).exp()       # [readings] Motor+
# Every pose's prior: the first known, the others only vaguely.
priors = concatenate([isotropic(*known_std)[None], isotropic(*vague_std) * np.ones(n_steps)])  # [poses] Twist+ <- Forque-
# What each reading and each anchor weighs: the inverse of its covariance, a quadratic form on twists.
weights = noises.inverse()                                                      # [readings] Forque- <- Twist+
anchor_weights = priors.inverse()                                               # [poses] Forque- <- Twist+

## 2. Dead reckoning

Dead reckoning chains the step readings from the known start and drifts across the lap, and its uncertainty grows with it: each step's noise, carried to the world frame by the pose where it enters, adds to all the noise before it, and the sum, carried back to each pose's own frame, is that pose's covariance. Every pose after the first is anchored with its vague prior at its dead-reckoned pose. In the figure, the true path is grey, the dead-reckoned trajectory and its two-standard-deviation confidence ellipsoids are orange, and the loop-closing reading links the start to the last pose in green.

In [ ]:
def reckon(poses: Motor, noises: Covariance, priors: Covariance) -> Covariance:
    """Under dead reckoning each pose's covariance is the first pose's prior grown by the noise of every step before it."""
    step_noises = concatenate([priors[:1], noises[:len(poses) - 1]])            # [poses] Twist+ <- Forque-
    world_uncertainty = (poses >> step_noises(poses << Forque)).cumsum(axis=0)  # [poses] Twist+ <- Forque-
    return poses << world_uncertainty(poses >> Forque)                          # [poses] Twist+ <- Forque-


def ellipsoids(poses: Motor, uncertainty: Covariance, sigmas: float) -> Quadric:
    """The quadric of each pose's position vanishes on its confidence ellipsoid, `sigmas` standard deviations out."""
    positions = poses >> origin                                                 # [...] Point-
    # A twist moves the position by its commutator with it: a map from twists to displacements.
    shift = Twist.commutator(positions)(poses >> Twist)                         # [...] Point- <- Twist+
    # Its adjugate pulls each plane back to the forque that reads the same displacement from the twist.
    readout = shift.adjugate()                                                  # [...] Forque- <- Plane+
    # The covariance pushed forward through the motion adds to the position's own dyad: its second moment.
    moment = positions * (Plane & positions) + shift(uncertainty(readout))      # [...] Point- <- Plane+
    return moment.inverse() - (1 + sigmas**2) * mv.w * (mv.w & Point)           # [...] Plane+ <- Point-


# The step readings, chained from the known start.
dead_reckoning = concatenate([truth[:1], truth[0] * readings[:-1].reverse().cumprod(axis=0).reverse()])  # [poses] Motor+
# Every pose's anchor: the first where it is known, the others where dead reckoning puts them.
anchors = concatenate([truth[:1], dead_reckoning[1:]])                          # [poses] Motor+
reckoned_uncertainty = reckon(dead_reckoning, noises, priors)                   # [poses] Twist+ <- Forque-
reckoned_ellipsoids = ellipsoids(dead_reckoning, reckoned_uncertainty, sigmas)  # [poses] Plane+ <- Point-
save_figure(render.draw_lap(truth, dead_reckoning, reckoned_ellipsoids, dead_reckoning[[tails[-1], heads[-1]]]), "odometry_dead_reckoning");

## 3. Mismatches and their pull

A reading's mismatch is the twist from what it read to the relative pose the poses imply, `(readings.inverse() * relative_motors).log() * 2`. Weighted by the reading's information it becomes a forque that pulls on the reading's head, while the opposite forque, carried back across the relative motor, pulls on its tail; the anchors pull likewise. The sum of those pulls is the gradient of the objective. The same pull, applied to the change a candidate correction makes to each reading, is the Gauss-Newton curvature, so the operator is never assembled as a matrix, and what a pose's perturbation alone causes, its diagonal block, preconditions the solve.

In [ ]:
def pull(relative_motors: Motor, reading_twists: Twist, anchor_twists: Twist) -> Forque:
    """The forque on every pose is the sum of those its readings' and its anchor's twists exert on it."""
    head_forques = weights(reading_twists)                                      # [..., readings] Forque-
    anchor_forques = anchor_weights(anchor_twists)                              # [..., poses] Forque-
    # A reading pulls on its head, and on its tail the opposite way carried across the relative motor.
    tail_forques = -(relative_motors >> head_forques)                           # [..., readings] Forque-
    pose_forques = anchor_forques.at[..., heads].add(head_forques)              # [..., poses] Forque-
    return pose_forques.at[..., tails].add(tail_forques)                        # [..., poses] Forque-


def gradient(poses: Motor) -> Forque:
    """The gradient of the pose-graph objective is the forque its mismatches exert on every pose."""
    relative_motors = poses[tails].inverse() * poses[heads]                     # [readings] Motor+
    reading_mismatches = (readings.inverse() * relative_motors).log() * 2       # [readings] Twist+
    anchor_mismatches = (anchors.inverse() * poses).log() * 2                   # [poses] Twist+
    return pull(relative_motors, reading_mismatches, anchor_mismatches)         # [poses] Forque-


def curvature(poses: Motor, twists: Twist) -> Forque:
    """The Gauss-Newton information operator takes a twist perturbation of every pose to the forques it causes."""
    # Each reading measures the twist at its head minus the twist at its tail carried to the head.
    relative_motors = poses[tails].inverse() * poses[heads]                     # [readings] Motor+
    reading_twists = twists[..., heads] - (relative_motors << twists[..., tails])  # [..., readings] Twist+
    return pull(relative_motors, reading_twists, twists)                        # [..., poses] Forque-


def curvature_alone(poses: Motor) -> Information:
    """The per-pose diagonal block of the Gauss-Newton information operator is what a pose's perturbation alone causes."""
    relative_motors = poses[tails].inverse() * poses[heads]                     # [readings] Motor+
    # Moving a reading's head alone meets the reading's weight directly; moving its tail alone
    # meets the weight carried back across the relative motor.
    tail_weights = relative_motors >> weights(relative_motors << Twist)         # [readings] Forque- <- Twist+
    pose_weights = anchor_weights.at[heads].add(weights)                        # [poses] Forque- <- Twist+
    return pose_weights.at[tails].add(tail_weights)                             # [poses] Forque- <- Twist+

## 4. The solvers

Conjugate gradients solve the normal equations for the twist corrections, preconditioned by each pose's diagonal block, across any number of leading batch axes; they converge in at most as many steps as there are scalar unknowns. The objective is nonlinear in the poses, so Gauss-Newton solves the linearised system at the current poses and applies the correction as a motor on the right of each pose.

In [ ]:
def conjugate_gradients(poses: Motor, forques: Forque) -> Iterator[Twist]:
    """Preconditioned conjugate gradients yield successive twist corrections converging on the solution of the Gauss-Newton normal equations."""
    preconditioner = curvature_alone(poses)                                     # [poses] Forque- <- Twist+
    correction = 0 * preconditioner.solve(forques)                              # [..., poses] Twist+
    residual = forques                                                          # [..., poses] Forque-
    direction = preconditioner.solve(residual)                                  # [..., poses] Twist+
    residual_energy = (residual & direction).sum(axis=-1)                       # [...] Scalar+
    # Conjugate gradients converge in at most as many steps as there are scalar unknowns.
    for _ in range(len(Twist.output_subspace) * len(poses)):
        direction_forques = curvature(poses, direction)                         # [..., poses] Forque-
        step = (residual_energy / (direction_forques & direction).sum(axis=-1))[..., None]
        correction = correction + direction * step                              # [..., poses] Twist+
        residual = residual - direction_forques * step                          # [..., poses] Forque-
        preconditioned_residual = preconditioner.solve(residual)                # [..., poses] Twist+
        next_energy = (residual & preconditioned_residual).sum(axis=-1)         # [...] Scalar+
        direction = preconditioned_residual + direction * (next_energy / residual_energy)[..., None]  # [..., poses] Twist+
        residual_energy = next_energy
        yield correction


def gauss_newton(poses: Motor, damping: float, n_iterations: int) -> Iterator[Motor]:
    """Damped Gauss-Newton iterations yield the poses after each step."""
    for _ in range(n_iterations):
        *_, correction = conjugate_gradients(poses, -gradient(poses))           # [poses] Twist+
        poses = poses * (correction * (0.5 * damping)).exp()                    # [poses] Motor+
        yield poses

## 5. The most likely poses and their uncertainty

Three full Gauss-Newton steps from dead reckoning reach the most likely poses. Each pose's marginal covariance is its block of the inverse of the information operator: every unit mode of reading and anchor noise, solved for in one batch, gives the pose errors it induces, and their outer products, summed at each pose, are its covariance. In the figure, the recovered poses and their confidence ellipsoids are drawn in blue over the grey true path and the dashed orange dead-reckoned trajectory.

In [ ]:
n_full_steps = 3

In [ ]:
def marginals(poses: Motor) -> Covariance:
    """Each pose's marginal covariance sums the pose errors that every unit mode of reading and anchor noise induces."""
    relative_motors = poses[tails].inverse() * poses[heads]                     # [readings] Motor+
    noise_modes = concatenate([modes(noises), modes(priors)])                   # [sites, modes] Twist+
    unit_modes = noise_modes[..., None] * np.eye(len(noise_modes))[:, None]     # [sites, modes, sites] Twist+
    mode_forques = pull(relative_motors, unit_modes[..., :len(tails)], unit_modes[..., len(tails):])  # [sites, modes, poses] Forque-
    *_, pose_errors = conjugate_gradients(poses, mode_forques)                  # [sites, modes, poses] Twist+
    return (pose_errors * (pose_errors & Forque)).sum(axis=(0, 1))              # [poses] Twist+ <- Forque-


*_, poses = gauss_newton(dead_reckoning, 1.0, n_full_steps)                     # [poses] Motor+
uncertainty = marginals(poses)                                                  # [poses] Twist+ <- Forque-
save_figure(render.draw(truth, dead_reckoning, reckoned_ellipsoids, poses, ellipsoids(poses, uncertainty, sigmas)), "odometry");

## 6. Closing the lap

When damped, Gauss-Newton moves the poses by a fraction of each correction and relinearises at the updated poses. Each step removes that fraction of the remaining pose error, shrinking the excess covariance above the posterior marginal covariance by `(1 - damping)**2`. Stepping one fifth of the way at a time from dead reckoning closes the loop and contracts every pose's ellipsoid onto the posterior uncertainty.

In [ ]:
# Three tenths of each correction per step.
damping = 0.3
n_damped_steps = 10
duration_ms = 333

In [ ]:
def uncertainties(iterates: Iterator[Motor], uncertainty: Covariance, damping: float) -> Iterator[tuple[Motor, Covariance]]:
    """Across damped Gauss-Newton steps the poses' covariance relaxes towards their marginals; the steps yield the poses with it."""
    for poses in iterates:
        posterior = marginals(poses)                                            # [poses] Twist+ <- Forque-
        uncertainty = posterior + (uncertainty - posterior) * (1 - damping)**2  # [poses] Twist+ <- Forque-
        yield poses, uncertainty


# The poses and their uncertainty after each damped step.
closing_steps = uncertainties(gauss_newton(dead_reckoning, damping, n_damped_steps), reckoned_uncertainty, damping)
show_animation(render.animate(truth, dead_reckoning, reckoned_ellipsoids,
                              ((poses, ellipsoids(poses, uncertainty, sigmas)) for poses, uncertainty in closing_steps)),
               "odometry", duration_ms, stepped=True)

In [ ]:
# checks
# The gradient falls a millionfold from dead reckoning to the most likely poses.
dead_gradient = np.abs(gradient(dead_reckoning).kernel).max()
np.testing.assert_allclose(gradient(poses).kernel, 0.0, atol=1e-6 * dead_gradient)